Deleting_Clients_History.ipynb


    Основные Функции:
        Удаление Всей истории клиентов по списку торговых счетов;
        Удаление счетов по списку;

    Дополнительные Функции:
        Загрузка список счетов МТ5 из файла;
        Удаление нулевого пополнения;
        Подсчёт сумм по определённым колонкам массива;
        Массовая проверка / исправление балансов и кредитов;
        Подсчёт сумм по списку колонок массива;
        Подсчёт сумм по определённой колонке массива;
    
    

In [ ]:
import MT5Manager                           # подключаем библиотеку
manager = MT5Manager.ManagerAPI()           # создаем менеджерский интерфейс
admin = MT5Manager.AdminAPI()               # создаем администраторский интерфейс
#import datetime                            # подключаем библиотеку для работы с датами
import pandas as pd                         # Чтение файла с содержимым дата фрейма
import os
#import time
import numpy as np

server_mt5_ip_port = "1**.*6.*4.2**:4**"
manager_mt5_login = *******
manager_mt5_password = "*******"

# Функция поиска позиций заданных значений в массиве <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# `````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
def positions_in_array(array, values_to_find):
    """Поиск позиций заданных значений в массиве.
    :param array: Двумерный массив, в котором выполняется поиск
    :param values_to_find: Список значений для поиска
    :return: Словарь {значение: [(строка, позиция в строке), ...]}"""
    result = {value: [] for value in values_to_find}  # Инициализируем словарь с пустыми списками
    for i, row in enumerate(array):
        for j, element in enumerate(row):
            if element in values_to_find:
                result[element].append((i, j))  # Добавляем (строка, позиция) в список соответствующего значения
    return result
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Извлекает значения из указанной колонки структурированного массива <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
def get_column_values(array, column_index):
    """Извлекает значения из указанной колонки структурированного массива.
    Parameters:
        array (numpy.ndarray): Исходный массив.
        column_index (int): Индекс колонки (начиная с 0).
    Returns:
        numpy.ndarray: Значения указанной колонки."""
    if not array.dtype.names:
        raise ValueError("Массив не имеет структурированных данных.")
    
    field_name = array.dtype.names[column_index]
    return array[field_name]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Функция для настройки отображения массива <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
def np_set_printoptions(name, array, num, thres_hold=10):                       # Устанавливаем опции отображения для всей библиотеки NumPy
    np.set_printoptions(threshold=thres_hold)
    if isinstance(array, np.ndarray):                                           # Вывод названия и содержимого массива
        print(f"{name} {len(array) if array.size > 0 else 0}:\n", array)

        first_elements = [deal[num] for deal in array]
        first_elements = list(map(int, first_elements))
        print("first_elements = ", first_elements)

    else:
        print(f"{name} не является массивом NumPy. Значение: {array}")          # Если array не массив, выводим сообщение
        first_elements =0
    return first_elements
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Функция подсчёта суммы по колонкам МАССИВА переданным списком <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
def sum_columns_by_indices(array, indices):
    """
    Вычисляет сумму значений для указанных колонок в массиве или списке кортежей и общую сумму.
    :param array: Одномерный массив NumPy или список кортежей
    :param indices: Список индексов колонок, значения которых нужно суммировать
    :return: Словарь с суммами по колонкам и общая сумма
    """
    column_sums = {}
    total_sum = 0

    for index in indices:                                    # Проходим по индексам колонок
        column_sum = sum(row[index] for row in array)        # Суммируем значения по данному индексу в строках
        column_sums[index] = column_sum
        total_sum += column_sum

    return {"column_sums": column_sums, "total_sum": total_sum}
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

In [ ]:

        value_to_find = 1.11
        found_rows = []
        positions = []

        for i, row in enumerate(array):
            for j, element in enumerate(row):
                if element == value_to_find:
                    positions.append((i, j))  # Добавляем индекс строки и позиции в строке
                    found_rows.append(row)
                    break  # Можно выйти, если нужно только первое совпадение в строке

        print("Позиции значения в массиве (строка, позиция в строке):", positions)
        print("Строки, содержащие значение:")
        for row in found_rows:
            print(row)

In [ ]:
print("Создаём DF с торговыми счетами, для переноса в МТ5")

file_path = 'files/customers_accounts.csv'     
if os.path.exists(file_path): print(f"Файл '{file_path}' существует.")      # Проверяем, существует ли файл
else: print(f"Файл '{file_path}' не найден.")

pd.set_option('display.max_columns', None)                                  # Устанавливаем настройку для отображения
pd.set_option('display.max_rows', 5)                                        # Устанавливаем настройку для отображения
trading_acc_df = pd.read_csv(file_path, sep=',')                            # Создаем DataFrame из CSV-файла
trading_acc_df['datetime'] = pd.to_datetime(trading_acc_df['created_at'])   # Преобразуем строковые значения в объекты datetime
trading_acc_df['unix_time'] = trading_acc_df['datetime'].apply(lambda x: int(x.timestamp()))    # Преобразуем datetime в Unix-время (в секундах) без миллисекунд
display("Дата фрейм с отформатированными данными:", trading_acc_df)

login_list = list(trading_acc_df["account_id"])                             # Запрашиваем список логинов из ДФ

print(f"Список Логинов к зачистке истории: \n login_list = {login_list}")

In [ ]:
# Загружаем список счетов МТ5 из файла <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_path = 'files/acc_list.csv'
with open(file_path, 'r') as file:                                          # Читаем файл с идентификаторами
    account_ids = file.read().strip().split(', ')                           # Извлекаем строки и делим их по запятой
account_ids = [int(id.strip()) for id in account_ids]                       # Преобразуем идентификаторы в целые числа
print(f"Длинна списка счетов МТ5 из файла {file_path}: {len(account_ids)}")
#account_ids = account_ids[:3]  # Оставляем только первые три элемента # Уже преобразованы в целые числа, повторно преобразовывать не нужно

    |
    Поэтапное удаление балансовых сделок с определёнными критериями <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

    ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
    Полная Зачистка истории Пользователей одним блоком <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

In [ ]:
#login_list = account_ids #                                                                                # Определяем список торговых счетов
login_list = [254860]#, 242047, 237778, 239045, 237663]
print("Группировка строк для извлечения идентификаторов для удаления \n Подключение Администратора МТ5 = ",
      admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password))

order_array = admin.OrderRequestByLoginsNumPy(login_list)                                                           # Получаем список ОРДЕРОВ
first_elements_order = np_set_printoptions(" \n Массив ОРДЕРОВ к удалению", order_array, 0, 3)

deal_array = admin.DealRequestByLoginsNumPy(login_list, 307718788, 1759407988)
first_elements_deal = np_set_printoptions(" \n Массив СДЕЛОК к удалению", deal_array, 0, 35)                         # Получаем список СДЕЛОК
admin.Disconnect()

In [ ]:
admin.Disconnect()

In [ ]:
#print(deal_array[1])


class Example:
    def __init__(self):
        self.name = "Test Object"
        self.value = 42
        self.is_active = True

# Создание экземпляра объекта
obj = deal_array

# Получение всех атрибутов объекта
attributes = dir(obj)

# Фильтрация пользовательских атрибутов (без методов и служебных)
#user_attributes = [attr for attr in attributes if not attr.startswith('__') and not callable(getattr(obj, attr))]

# Вывод атрибутов и их значений
#print(deal_array[1].Login)
#print(obj.Login())
obj = deal_array[1]

# Печать полей и их значений
if hasattr(obj, "dtype") and obj.dtype.names:
    for field_name in obj.dtype.names:
        value = obj[field_name]
        print(f"Field: {field_name}, Value: {value}")
else:
    print("The object does not have structured fields.")


# Получаем список полей и их типов
fields = deal_array.dtype.fields
# Выводим имена полей и их типы
for field_name, (field_type, _) in fields.items():
    print(f"Поле: {field_name}, Тип: {field_type}")



In [ ]:
import numpy as np

def array_to_dataframe_with_multiline_headers(array):

    import pandas as pd
    from IPython.display import display
    """
    Преобразует структурированный numpy-массив в pandas DataFrame.
    Заголовки столбцов включают имя параметра (первая строка) и его формат данных (вторая строка).
    :param array: numpy.ndarray со структурированным dtype
    :return: pandas.DataFrame
    """
    if not isinstance(array, np.ndarray) or not array.dtype.names:
        raise ValueError("Ожидается структурированный numpy.ndarray")
    
    # Формируем заголовки с переносами строк
    columns_with_multiline = [
        f"{field}<br>({str(array.dtype[field])})" for field in array.dtype.names
    ]
    df = pd.DataFrame(array.tolist(), columns=columns_with_multiline)
    
    return df

df = array_to_dataframe_with_multiline_headers(deal_array)

styled_df = df.style.set_table_styles(
    [{'selector': 'th', 'props': [('white-space', 'pre-wrap'), ('text-align', 'center')]}]
)

# Отображение DataFrame с использованием HTML
from IPython.display import HTML
display(HTML(df.to_html(escape=False)))

In [ ]:
# Получение списка ПОЗИЦИЙ к Удалению <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
position_array = admin.PositionRequestByLoginsNumPy(login_list)
if position_array is not None and isinstance(position_array, np.ndarray):
    # Удаление поля 'TimeCreateMsc'
    fields = list(position_array.dtype.names)                       # Получаем список всех полей
    fields.remove('TimeCreateMsc')                                  # Удаляем НЕ нужное поле
    position_array_2 = position_array[fields]                       # Создаем новый массив только с нужными полями
    first_elements_position = np_set_printoptions(" \n Массив ПОЗИЦИЙ к удалению", position_array_2, 19, 5)         # Получаем список ПОЗИЦИЙ
    print(first_elements_position)
else: print(f"Массив ПОЗИЦИЙ к удалению отсутствует; position_array = {position_array}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

In [ ]:
# Подсчёт сумм по списку колонок массива <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# #```````````````````````````````````````````````````````````````````````````````````````````````````` 
array = deal_array
values_to_find = [-33.06]

result = positions_in_array(array, values_to_find)
print("Результаты поиска:")
for value, positions in result.items():
    print(f"Значение {value}: позиции {positions}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

In [ ]:
# Получение списка значений из колонки массива <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# #``````````````````````````````````````````````````````````````````````````````````````````````````````````
index = 15
column_values = get_column_values(array, index)
print(column_values)

In [ ]:
# Подсчёт сумм по определённой колонке массива <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# #```````````````````````````````````````````````````````````````````````````````````````````````````` 
indices = [38, 15, 16, 14]                             # Fee[38], Swap[15], Commission[16], Profit[14]
result = sum_columns_by_indices(deal_array, indices)
print("Суммы по колонкам:", result["column_sums"])
print("Общая сумма:", result["total_sum"])
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Balans H 4647.15          Можно получить только из Диалс
# Balans O 4214.57

# Balance $ 4397.15 CRM
# Credit 250


In [8]:
# Загружаем список счетов МТ5 из файла <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_path = 'files/acc_list.csv'
with open(file_path, 'r') as file:                                          # Читаем файл с идентификаторами
    account_ids = file.read().strip().split(', ')                           # Извлекаем строки и делим их по запятой
account_ids = [int(id.strip()) for id in account_ids]                       # Преобразуем идентификаторы в целые числа
print(f"Длинна списка [account_ids] счетов МТ5 из файла {file_path}: {len(account_ids)}")

Длинна списка [account_ids] счетов МТ5 из файла files/acc_list.csv: 1566


In [7]:
import os

# Путь к файлу с идентификаторами счетов
file_path = r'C:\unique_data\rep_fo_metatrader_server\files\acc_list.csv'

# Проверка на существование файла
if os.path.exists(file_path):
    # Чтение файла с идентификаторами
    with open(file_path, 'r') as file:
        account_ids = file.read().strip().split(', ')  # Извлекаем строки и делим их по запятой
    
    # Преобразование идентификаторов в целые числа
    account_ids = [int(id.strip()) for id in account_ids]
    
    # Вывод длины списка и сам список для проверки
    print(f"Длина списка счетов МТ5 из файла {file_path}: {len(account_ids)}")
    print(account_ids)
else:
    print(f"Файл {file_path} не найден")

Длина списка счетов МТ5 из файла C:\unique_data\rep_fo_metatrader_server\files\acc_list.csv: 1566
[262146, 262151, 73740, 229393, 237585, 237591, 237602, 163878, 229419, 237613, 221255, 262218, 237646, 237656, 237657, 114780, 237660, 237678, 65649, 237689, 262266, 73852, 237694, 90243, 262291, 262298, 262302, 237728, 262308, 262317, 262320, 237747, 237748, 262324, 237757, 262335, 237764, 262340, 180429, 237776, 237778, 237779, 237796, 237797, 262391, 262399, 262411, 237841, 262422, 262424, 196889, 237854, 147748, 237870, 237871, 237878, 237895, 237904, 262480, 262483, 262490, 82283, 180587, 237940, 262523, 262524, 57730, 74114, 262540, 262552, 164260, 262570, 262572, 238002, 238004, 246204, 238013, 254397, 262593, 254406, 262604, 262606, 254419, 238048, 238056, 238068, 205301, 238070, 262647, 238078, 262655, 254464, 238081, 238097, 262681, 115229, 139807, 115232, 238111, 238114, 41508, 254500, 238119, 238122, 254508, 221741, 238136, 238138, 262720, 238145, 238155, 238163, 238165, 23817

In [8]:
#account_ids = [218624, 237569, 235525, 256008, 101385, 233993, 43532, 143886, 192527, 210959, 112145, 94739, 237080, 7194, 238625, 96306, 97843, 86067, 182835, 85046, 238644, 234035, 102457, 200762, 237627, 70714, 237118, 237119, 225344, 238655, 50754, 212547, 219203, 231493, 237129, 117322, 237131, 237132, 249420, 58448, 69200, 239701, 197717, 237658, 57435, 204892, 247900, 237663, 184419, 199268, 217701, 237670, 241765, 238702, 124529, 183419, 237691, 241787, 216191, 198274, 227970, 145027, 196743, 238729, 238217, 240779, 237710, 222868, 239253, 233110, 230039, 235671, 149667, 226468, 206499, 235006, 127656, 239277, 69806, 88241, 220338, 242355, 221874, 178869, 238262, 207544, 53947, 115904, 181953, 215234, 255170, 240838, 68295, 33993, 54989, 102606, 231117, 238800, 238292, 208085, 159445, 206039, 136921, 82138, 237787, 9948, 243419, 242399, 179936, 249058, 205027, 233188, 234213, 235748, 242915, 180969, 204521, 235755, 80620, 241901, 248048, 176370, 233209, 236282, 211194, 91902, 186111, 235265, 234754, 235267, 236292, 239875, 223494, 236294, 232713, 211211, 185101, 242446, 233231, 233233, 162580, 233237, 93462, 237335, 233240, 184600, 84761, 256283, 182557, 235296, 202530, 233254, 107303, 235819, 55596, 249131, 234798, 239407, 238384, 85806, 237875, 240948, 233272, 240962, 236868, 110405, 239430, 249159, 217416, 98124, 227148, 86354, 237395, 235860, 217943, 235351, 163161, 240986, 161625, 192864, 13665, 200034, 199523, 249187, 87405, 165232, 242545, 256369, 149875, 17782, 117622, 167288, 222590, 255871, 201601, 238467, 157572, 206724, 75140, 236932, 238476, 244630, 179097, 71582, 237471, 91552, 51617, 228261, 169893, 236970, 117167, 239025, 248754, 74174, 97726, 261054, 134593, 75203, 232388, 143301, 231878, 200647, 106952, 239045, 152524, 168912, 212945, 238034, 236499, 212436, 234970, 169436, 83421, 240609, 169448, 82923, 64492, 228334, 186863, 238062, 236014, 238577, 236017, 115190, 235002, 238075, 104956, 239101, 103422]
#login_list = [233231]
account_ids = [267205]

In [10]:
# Полная Зачистка истории Пользователей <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
login_list = account_ids #                                                                                # Определяем список торговых счетов
print(f"Счетов к зачистке истории: {len(login_list)}")
print("Группировка строк для извлечения идентификаторов для удаления \n Подключение Администратора МТ5 = ",
    admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password))

order_array = admin.OrderRequestByLoginsNumPy(login_list)                                                           # Получаем список ОРДЕРОВ
first_elements_order = np_set_printoptions(" \n Массив ОРДЕРОВ к удалению", order_array, 0, 3)

deal_array = admin.DealRequestByLoginsNumPy(login_list, 307718788, 1759407988)
first_elements_deal = np_set_printoptions(" \n Массив СДЕЛОК к удалению", deal_array, 0, 3)                         # Получаем список СДЕЛОК

# Получение списка ПОЗИЦИЙ к Удалению <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
position_array = admin.PositionRequestByLoginsNumPy(login_list)
if position_array is not None and isinstance(position_array, np.ndarray):
    # Удаление поля 'TimeCreateMsc'
    fields = list(position_array.dtype.names)                       # Получаем список всех полей
    fields.remove('TimeCreateMsc')                                  # Удаляем НЕ нужное поле
    position_array_2 = position_array[fields]                       # Создаем новый массив только с нужными полями
    first_elements_position = np_set_printoptions(" \n Массив ПОЗИЦИЙ к удалению", position_array_2, 27, 3)         # Получаем список ПОЗИЦИЙ
    print(first_elements_position)
else: print(f"Массив ПОЗИЦИЙ к удалению отсутствует; position_array = {position_array}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print("admin.Disconnect() = ", admin.Disconnect())
del deal_array

# Удаление ордеров <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("Список идентификаторов ОРДЕРОВ к удалению: \n", first_elements_order)
print(admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password))
orderl_delete_batch = admin.OrderDeleteBatch(first_elements_order)
#print(MT5Manager.LastError())
print("orderl_delete_batch = ", orderl_delete_batch)
if orderl_delete_batch:
    orderl_delete_batch = admin.DealDeleteBatch(first_elements_order)
    if orderl_delete_batch == False:
        print(f": {MT5Manager.LastError()}")
    else: orderl_delete_batch
else: print("\n Ордера к удалению отсутствуют")
admin.Disconnect()

# Удаление сделок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("Список идентификаторов сделок к удалению: ----------------------------------------\n", first_elements_deal)
print(admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password))
deal_delete_batch = admin.DealDeleteBatch(first_elements_deal)
#print(deal_delete_batch)
if deal_delete_batch == False:
    print(f": {MT5Manager.LastError()}")
else: deal_delete_batch
admin.Disconnect()

# Удаление позиций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
try:
    print("Список идентификаторов ПОЗИЦИЙ к удалению: \n", first_elements_position)
    print(admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password))

    position_delete_batch = admin.PositionDeleteBatch(first_elements_position)
    #print(position_delete_batch)
    if position_delete_batch == False:
        print(f": {MT5Manager.LastError()}")
    else: position_delete_batch
    admin.Disconnect()
except: print("Список идентификаторов ПОЗИЦИЙ к удалению: ОТСУТСТВУЕТ\n")

# Массовая проверка балансов и кредитов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("login_list к проверке: \n", login_list)
if admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password):
    #login_list = [233209, 233231]
    ubcb = admin.UserBalanceCheckBatch(login_list, 1)
    if  ubcb:
        print("\n проверка / исправление прошла успешно")
else:
    print(f"\n ERROR admin.Connect: {MT5Manager.LastError()}")
print("\n admin.Disconnect() = ", admin.Disconnect())

Счетов к зачистке истории: 1
Группировка строк для извлечения идентификаторов для удаления 
 Подключение Администратора МТ5 =  True
 
 Массив ОРДЕРОВ к удалению не является массивом NumPy. Значение: False
 
 Массив СДЕЛОК к удалению 103:
 [(4267738, b'', 267205, 1100, 0, 5, 0, 2, 2,      0., 1685577600, b'',  0.     ,    0, 1.00e+08, 0., 0., 0., 0.   , 0.     , 0,       0, b'100000000', 0., 0., 0, 0.    , 0., 0, 1685577600000, 2, b'', 0.,  32, 0., 0.,        0, 0, 0., 0.,  0.     ,  0.     , 0.)
 (4267739, b'', 267205, 1100, 0, 2, 0, 2, 2,      0., 1736507476, b'',  0.     ,    0, 2.45e+02, 0., 0., 0., 0.   , 0.     , 0,       0, b'#61218, balance, deposit', 0., 0., 0, 0.    , 0., 0, 1736507476000, 2, b'', 0.,  32, 0., 0.,        0, 0, 0., 0.,  0.     ,  0.     , 0.)
 (4267740, b'', 267205,    0, 0, 0, 0, 5, 2, 100000., 1736515747, b'EURUSD',  1.03041,  100, 0.00e+00, 0., 0., 0., 1.   , 1.03041, 0, 4587324, b'#200402, RP=1.0, SP=0.0', 0., 0., 0, 0.    , 0., 0, 1736515747854, 0, b'', 0.

In [3]:
print("\n admin.Disconnect() = ", admin.Disconnect())


 admin.Disconnect() =  True


In [5]:
print(login_list)

[215034, 194863, 207593, 226797, 110677, 235692]


In [17]:
#login_list = [267136, 267521, 267522, 267655, 267917, 239633, 267538, 267539, 267666, 267922, 267542, 267670, 267161, 267673, 267419, 267674, 267549, 265762, 207277, 267569, 267194, 257979, 267707, 267965, 266942, 265792, 265666, 262218, 267725, 234705, 265426, 267857, 267348, 267480, 267097, 267356, 267619, 267875, 267380, 267769, 267772]

In [6]:
# Удаление счетов по списку <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````
print(f"Список счетов к удалению: [{len(login_list)}] \n", login_list, "\n")
#admin = admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password) # Подключаемся к серверу
if admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password):      # Подключаемся к серверу
    true_delete_acc  = []
    false_delete_acc = []
    for i in login_list:
        if admin.UserDelete(i):
            #print(f"счёт [{i}] удалён")
            true_delete_acc.append(i)
        else:
            print(f"EEEOR счёт [{i}] НЕ удалён {MT5Manager.LastError()}")
            false_delete_acc.append(i)
    admin.Disconnect()
    print(f"\n Успешно удалённые [{len(true_delete_acc)}] торговые счета: \n {true_delete_acc}")
    print(f"\n НЕ далось удалить [{len(false_delete_acc)}] счета: \n {false_delete_acc}")
else: print (f"ERROR проблемы с подключением {MT5Manager.LastError()}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

Список счетов к удалению: [255] 
 [218624, 237569, 235525, 256008, 101385, 233993, 43532, 143886, 192527, 210959, 112145, 94739, 237080, 7194, 238625, 215034, 96306, 97843, 86067, 182835, 85046, 238644, 234035, 102457, 200762, 237627, 70714, 237118, 237119, 225344, 238655, 50754, 212547, 219203, 231493, 237129, 117322, 237131, 237132, 249420, 70223, 58448, 69200, 239701, 197717, 110677, 237658, 57435, 204892, 247900, 237663, 184419, 199268, 217701, 237670, 241765, 238702, 124529, 183419, 237691, 241787, 216191, 198274, 227970, 145027, 196743, 238729, 238217, 240779, 237710, 222868, 239253, 233110, 230039, 235671, 149667, 226468, 206499, 235006, 127656, 235692, 239277, 69806, 88241, 220338, 242355, 221874, 178869, 238262, 207544, 53947, 115904, 181953, 215234, 255170, 240838, 68295, 33993, 54989, 102606, 231117, 238800, 238292, 208085, 159445, 206039, 136921, 82138, 237787, 9948, 243419, 242399, 179936, 249058, 205027, 233188, 234213, 235748, 242915, 180969, 204521, 235755, 207593, 2419

In [26]:
login_list = [30006,303002,303060,234705,207277,239633,267453,239045,237663]

In [16]:
login_list = account_ids # 
print(f"Длинна списка счетов МТ5 из файла {file_path}: {len(login_list)}")

Длинна списка счетов МТ5 из файла files/acc_list.csv: 12


In [17]:
# Удаление нулевого пополнения <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````
print(f"Список счетов к удалению: [{len(login_list)}] \n", login_list, "\n")
def np_set_printoptions_2(name, array, num, data, summ, thres_hold=10):         # Функция для настройки отображения массива с фильтрацией по 10-й  и 14 позиции
    np.set_printoptions(threshold=thres_hold)                                   # Устанавливаем опции отображения для всей библиотеки NumPy
    if isinstance(array, np.ndarray):                                           # Вывод названия и содержимого массива
        print(f"{name} {len(array) if array.size > 0 else 0}:\n", array)
        first_elements = []                                                        
        for deal in array:
            try:
                print(deal[22])
                if deal[10] == data and  deal[14] == summ:             # Проверяем, что значение в 10-й позиции равно 1685577600
                    first_elements.append(int(deal[num]))                       # Преобразуем указанный элемент в int и добавляем в список
            except (ValueError, IndexError, TypeError) as e:
                print(f"Ошибка обработки элемента: {deal}, {e}")
        print("first_elements = ", first_elements)
    else:
        print(f"ERROR!: {name} не является массивом NumPy. Значение: {array}")          # Если array не массив, выводим сообщение
        first_elements = 0
    return first_elements


"""account_ids = [266387]
login_list = [266387]
print(type(login_list))"""
num  = 0                                              # Позиция в массиве по которой формируется список
data = 1685577600                                     # Точная дата и время 
summ = 100000000                                      # Cума транзакции 

print("Группировка строк для извлечения идентификаторов для удаления \n Подключение Администратора МТ5 = ",
      admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password))
deal_array = admin.DealRequestByLoginsNumPy(login_list, 1685577600, 1685577601) 
first_elements_deal = np_set_printoptions_2(" \n Массив СДЕЛОК к удалению", deal_array, num, data, summ)

# Удаление сделок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("Список идентификаторов сделок к удалению: \n", first_elements_deal)
print(admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password))
deal_delete_batch = admin.DealDeleteBatch(first_elements_deal)

if deal_delete_batch == False:
    print(f": {MT5Manager.LastError()}")
else: deal_delete_batch
admin.Disconnect()

# Массовая проверка балансов и кредитов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("login_list к проверке: \n", login_list)
if admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password):
    #login_list = [233209, 233231]
    ubcb = admin.UserBalanceCheckBatch(login_list, 1)
    if  ubcb:
        print("\n проверка / исправление прошла успешно")
else:
    print(f"\n ERROR admin.Connect: {MT5Manager.LastError()}")
print("\n admin.Disconnect() = ", admin.Disconnect())

Список счетов к удалению: [12] 
 [254860, 238859, 262335, 234672, 235415, 234174, 225989, 75408, 241971, 97498, 246204, 262908] 

Группировка строк для извлечения идентификаторов для удаления 
 Подключение Администратора МТ5 =  True
 
 Массив СДЕЛОК к удалению 12:
 [(4264482, b'', 238859, 1100, 0, 5, 0, 2, 2, 0., 1685577600, b'', 0., 0, 1.e+08, 0., 0., 0., 0., 0., 0, 0, b'100000000', 0., 0., 0, 0., 0., 0, 1685577600000, 2, b'', 0., 32, 0., 0., 0, 0, 0., 0., 0., 0., 0.)
 (4264483, b'', 254860, 1100, 0, 5, 0, 2, 2, 0., 1685577600, b'', 0., 0, 1.e+08, 0., 0., 0., 0., 0., 0, 0, b'100000000', 0., 0., 0, 0., 0., 0, 1685577600000, 2, b'', 0., 32, 0., 0., 0, 0, 0., 0., 0., 0., 0.)
 (4264484, b'', 234672, 1100, 0, 5, 0, 2, 2, 0., 1685577600, b'', 0., 0, 1.e+08, 0., 0., 0., 0., 0., 0, 0, b'100000000', 0., 0., 0, 0., 0., 0, 1685577600000, 2, b'', 0., 32, 0., 0., 0, 0, 0., 0., 0., 0., 0.)
 ...
 (4264491, b'', 234174, 1100, 0, 5, 0, 2, 2, 0., 1685577600, b'', 0., 0, 1.e+08, 0., 0., 0., 0., 0., 0, 0

In [ ]:
#login_list = [242047]
print(f"Длинна списка счетов МТ5 из файла {file_path}: {len(login_list)}")

In [ ]:
# Массовая проверка балансов и кредитов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
print("login_list к проверке: \n", login_list)
if admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password):
    ubcb = admin.UserBalanceCheckBatch(login_list, 1)
    if ubcb: print("\n проверка / исправление прошла успешно")
else: print(f"\n ERROR admin.Connect: {MT5Manager.LastError()}")
print("\n admin.Disconnect() = ", admin.Disconnect())

In [ ]:
# Удаление одной сделки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print(admin.Connect(server_mt5_ip_port, manager_mt5_login, manager_mt5_password))
#print(admin.DealDelete(282711))
admin.Disconnect()

In [ ]:
admin.Disconnect()